<a href="https://colab.research.google.com/github/pikey-msc/AprendizMaquina/blob/main/2026-2/Bagging_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos de Ensamble: Bagging y Random Forest
## Curso Aprendizaje de Máquina — 8° Semestre Actuaría

---

## 1. Motivación: ¿Por qué ensamblar?

Sea $\hat{f}(x)$ un estimador con varianza $\sigma^2$. Si construimos $B$ estimadores **independientes** $\hat{f}_1, \dots, \hat{f}_B$ cada uno con varianza $\sigma^2$, el promedio $\bar{f}(x) = \frac{1}{B}\sum_b \hat{f}_b(x)$ tiene varianza:

$$\text{Var}\left(\bar{f}\right) = \frac{\sigma^2}{B}$$

El problema: los árboles entrenados sobre el **mismo** dataset tienen correlación $\rho > 0$, lo que modifica la varianza del promedio a:

$$\text{Var}\left(\bar{f}\right) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

Cuando $B \to \infty$, el segundo término desaparece pero el primero $\rho\sigma^2$ **no**. La estrategia de los métodos de ensamble es atacar $\rho$ (descorrelación) y/o $\sigma^2$ (reducción de varianza individual).

---

## 2. Bootstrap y la estimación OOB

### 2.1 Bootstrap

Sea $\mathcal{Z} = \{(x_i, y_i)\}_{i=1}^n$. Una muestra bootstrap $\mathcal{Z}^{*b}$ es de tamaño $n$ obtenida **con reemplazo** de $\mathcal{Z}$.

**Proposición.** La probabilidad de que una observación $i$ **no** sea seleccionada en ninguno de los $n$ sorteos es:

$$P(i \notin \mathcal{Z}^{*b}) = \left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0.368$$

Esto implica que en promedio el **36.8% de las observaciones quedan fuera** de cada muestra bootstrap — estas son las observaciones *Out-Of-Bag* (OOB).

### 2.2 Error OOB como estimador del error de generalización

Para cada $x_i$, los árboles que **no** usaron $i$ en su entrenamiento forman el conjunto $\mathcal{B}_i = \{b : i \notin \mathcal{Z}^{*b}\}$. El predictor OOB es:

$$\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} \hat{f}^{*b}(x_i)$$

Y el error OOB global es $\frac{1}{n}\sum_i L(y_i, \hat{y}_i^{\text{OOB}})$, que es un estimador **casi insesgado** del error de prueba (comparable a leave-one-out CV pero computacionalmente gratis).

---

## 3. Bagging (Bootstrap AGGregating)

**Algoritmo Bagging** (Breiman, 1996):

1. Para $b = 1, \dots, B$:
   - Muestrea $\mathcal{Z}^{*b}$ con reemplazo de $\mathcal{Z}$
   - Ajusta árbol profundo $\hat{f}^{*b}$ sobre $\mathcal{Z}^{*b}$ (sin poda)
2. Agrega:
   - **Regresión:** $\hat{f}_{bag}(x) = \frac{1}{B}\sum_{b=1}^B \hat{f}^{*b}(x)$
   - **Clasificación:** $\hat{C}_{bag}(x) = \text{mayoría}\{\hat{C}^{*b}(x)\}_{b=1}^B$

**Descomposición bias-varianza de Bagging:**

El bias del promedio no cambia respecto al árbol individual (árboles profundos ya tienen bias bajo), pero la varianza se reduce de $\sigma^2$ a $\rho\sigma^2 + \frac{(1-\rho)}{B}\sigma^2$. Para Bagging, como todos los árboles usan **todas** las variables, $\rho$ puede ser alto cuando hay variables dominantes.

---

## 4. Random Forest: Descorrelación activa

**Algoritmo Random Forest** (Breiman, 2001):

Idéntico a Bagging con una modificación clave: en **cada nodo** de cada árbol, se selecciona aleatoriamente un subconjunto de $m$ variables (de las $p$ totales) y la división solo puede usar esas $m$ variables.

- Clasificación: $m \approx \lfloor\sqrt{p}\rfloor$
- Regresión: $m \approx \lfloor p/3 \rfloor$

**Efecto en la correlación:** Al forzar que los árboles no puedan usar todas las variables, se rompe la dependencia entre ellos. Si hay una variable muy predictiva, en Bagging todos los árboles la usarán en el nodo raíz → correlación alta. Random Forest la excluye en fracción $\approx (p-m)/p$ de los nodos → reduce $\rho$ drásticamente.

**Teorema (Breiman, 2001).** El error de generalización de un Random Forest con $B \to \infty$ árboles converge a:

$$\text{PE}^* \leq \frac{\bar{\rho}(1 - s^2)}{s^2}$$

donde $s = E_x[\text{margen}(x)]$ es el margen esperado y $\bar{\rho}$ la correlación media entre árboles. El error está **acotado** y decrece al reducir $\bar{\rho}$ o aumentar el margen.

---

## 5. Importancia de Variables por Permutación

Para la variable $j$, en el árbol $b$:
1. Calcula el error OOB base: $e_b$
2. **Permuta** aleatoriamente los valores de la variable $j$ en el conjunto OOB
3. Recalcula el error con la variable permutada: $p_b$
4. Diferencia: $d_b^{(j)} = e_b - p_b$ (degradación por perder la variable $j$)

La importancia de la variable $j$ es:

$$v_j = \frac{\bar{d}^{(j)}}{s_{d^{(j)}}} = \frac{\frac{1}{B}\sum_b d_b^{(j)}}{\sqrt{\frac{1}{B-1}\sum_b (d_b^{(j)} - \bar{d}^{(j)})^2}}$$

Es un estadístico tipo $t$: valores grandes indican variables con alta importancia real (no mera correlación con $y$).

---
## 6. Setup y Carga de Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import gc

try:
    !rm -rf CursoDF
except: pass
!git clone --depth=1 "https://github.com/pikey-msc/CursoDF" 2>/dev/null
print('Listo')

In [ ]:
# Carga y preprocesamiento del dataset de crédito
credit = pd.read_csv('CursoDF/M3/Datos/credit.txt', sep=' ', header=0)
col_names = [
    'checking_status','duration','credit_history','purpose','credit_amount',
    'savings','employment','installment_rate','personal_status','debtors',
    'residence_since','property','age','other_plans','housing','existing_credits',
    'job','dependents','telephone','foreign_worker','target'
]
credit.columns = col_names
credit['target'] = credit['target'].astype(int)

# Encode categoricas de forma compacta
cat_cols = credit.select_dtypes('object').columns.tolist()
le = LabelEncoder()
for c in cat_cols:
    credit[c] = le.fit_transform(credit[c])

print(f'Shape: {credit.shape}  |  Target dist:\n{credit.target.value_counts(normalize=True).round(3)}')
del le; gc.collect()

---
## 7. Clase `PrepGener` (v2 — no modificar)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

class PrepGener(BaseEstimator, TransformerMixin):
    """Preprocesador genérico v2. No modificar."""
    def __init__(self, scaler='standard', imputer_strategy='mean',
                 balance=None, cat_cols=None):
        self.scaler = scaler
        self.imputer_strategy = imputer_strategy
        self.balance = balance
        self.cat_cols = cat_cols or []

    def fit(self, X, y=None):
        self.imputer_ = SimpleImputer(strategy=self.imputer_strategy)
        self.imputer_.fit(X)
        X_imp = self.imputer_.transform(X)
        scalers = {'standard': StandardScaler(), 'minmax': MinMaxScaler(),
                   'robust': RobustScaler(), None: None}
        self.scaler_ = scalers[self.scaler]
        if self.scaler_:
            self.scaler_.fit(X_imp)
        return self

    def transform(self, X, y=None):
        X_out = self.imputer_.transform(X)
        if self.scaler_:
            X_out = self.scaler_.transform(X_out)
        return X_out

    def fit_resample(self, X, y):
        self.fit(X, y)
        X_out = self.transform(X)
        if self.balance == 'smote':
            X_out, y = SMOTE(random_state=42).fit_resample(X_out, y)
        elif self.balance == 'under':
            X_out, y = RandomUnderSampler(random_state=42).fit_resample(X_out, y)
        return X_out, y

---
## 8. Split y Preprocesamiento

In [ ]:
X = credit.drop('target', axis=1).values
y = credit['target'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                            stratify=y, random_state=42)
prep = PrepGener(scaler='standard', balance=None)
X_tr_s, y_tr_s = prep.fit_resample(X_tr, y_tr)
X_te_s = prep.transform(X_te)
print(f'Train: {X_tr_s.shape}  Test: {X_te_s.shape}')
print(f'Distribución train: {np.bincount(y_tr_s)}')

---
## 9. Algoritmo Bagging desde cero

Implementación sin `sklearn.ensemble` para ilustrar el mecanismo:

In [ ]:
class BaggingScratch:
    """Bagging con árboles CART desde cero."""
    def __init__(self, n_estimators=50, max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.rng = np.random.default_rng(random_state)
        self.trees_, self.oob_indices_ = [], []

    def fit(self, X, y):
        n = len(y)
        self.classes_ = np.unique(y)
        self.oob_preds_ = np.zeros((n, len(self.classes_)))
        self.oob_counts_ = np.zeros(n)
        for _ in range(self.n_estimators):
            idx = self.rng.integers(0, n, n)           # bootstrap
            oob = np.setdiff1d(np.arange(n), idx)     # OOB
            tree = DecisionTreeClassifier(max_depth=self.max_depth,
                                          random_state=0)
            tree.fit(X[idx], y[idx])
            self.trees_.append(tree)
            if len(oob):
                proba = tree.predict_proba(X[oob])
                self.oob_preds_[oob] += proba
                self.oob_counts_[oob] += 1
        # OOB error
        mask = self.oob_counts_ > 0
        oob_hat = np.argmax(self.oob_preds_[mask] /
                            self.oob_counts_[mask, None], axis=1)
        self.oob_error_ = 1 - accuracy_score(y[mask],
                            self.classes_[oob_hat])
        return self

    def predict_proba(self, X):
        proba = np.mean([t.predict_proba(X) for t in self.trees_], axis=0)
        return proba

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


bag_scratch = BaggingScratch(n_estimators=100, max_depth=None, random_state=7)
bag_scratch.fit(X_tr_s, y_tr_s)
y_hat = bag_scratch.predict(X_te_s)
print(f'OOB error (scratch): {bag_scratch.oob_error_:.4f}')
print(f'Accuracy test (scratch): {accuracy_score(y_te, y_hat):.4f}')
print(f'AUC test (scratch): {roc_auc_score(y_te, bag_scratch.predict_proba(X_te_s)[:,1]):.4f}')

---
## 10. Algoritmo Random Forest desde cero

La única diferencia con Bagging: en cada nodo se muestrea un subconjunto de `max_features` variables.

In [ ]:
class RandomForestScratch:
    """Random Forest desde cero — solo usa numpy y DecisionTreeClassifier."""
    def __init__(self, n_estimators=100, max_features='sqrt',
                 max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_depth = max_depth
        self.rng = np.random.default_rng(random_state)
        self.trees_, self.feat_indices_ = [], []

    def _n_features(self, p):
        if self.max_features == 'sqrt': return max(1, int(np.sqrt(p)))
        if self.max_features == 'log2': return max(1, int(np.log2(p)))
        if isinstance(self.max_features, float): return max(1, int(self.max_features * p))
        return p

    def fit(self, X, y):
        n, p = X.shape
        m = self._n_features(p)
        self.classes_ = np.unique(y)
        self.oob_preds_ = np.zeros((n, len(self.classes_)))
        self.oob_counts_ = np.zeros(n)
        for _ in range(self.n_estimators):
            idx = self.rng.integers(0, n, n)
            oob = np.setdiff1d(np.arange(n), idx)
            feats = self.rng.choice(p, m, replace=False)
            self.feat_indices_.append(feats)
            tree = DecisionTreeClassifier(max_depth=self.max_depth,
                                          max_features=m, random_state=0)
            tree.fit(X[np.ix_(idx, feats)], y[idx])
            self.trees_.append(tree)
            if len(oob):
                proba = tree.predict_proba(X[np.ix_(oob, feats)])
                self.oob_preds_[oob] += proba
                self.oob_counts_[oob] += 1
        mask = self.oob_counts_ > 0
        oob_hat = np.argmax(self.oob_preds_[mask] /
                            self.oob_counts_[mask, None], axis=1)
        self.oob_error_ = 1 - accuracy_score(y[mask],
                            self.classes_[oob_hat])
        return self

    def predict_proba(self, X):
        probas = np.zeros((len(X), len(self.classes_)))
        for tree, feats in zip(self.trees_, self.feat_indices_):
            probas += tree.predict_proba(X[:, feats])
        return probas / self.n_estimators

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def feature_importance_permutation(self, X, y):
        """Importancia por permutación sobre cualquier conjunto."""
        base_acc = accuracy_score(y, self.predict(X))
        importances = np.zeros(X.shape[1])
        rng = np.random.default_rng(0)
        for j in range(X.shape[1]):
            X_perm = X.copy()
            X_perm[:, j] = rng.permutation(X_perm[:, j])
            importances[j] = base_acc - accuracy_score(y, self.predict(X_perm))
        del X_perm; gc.collect()
        return importances


rf_scratch = RandomForestScratch(n_estimators=100, max_features='sqrt',
                                  max_depth=None, random_state=7)
rf_scratch.fit(X_tr_s, y_tr_s)
y_hat_rf = rf_scratch.predict(X_te_s)
print(f'OOB error (RF scratch): {rf_scratch.oob_error_:.4f}')
print(f'Accuracy test (RF scratch): {accuracy_score(y_te, y_hat_rf):.4f}')
print(f'AUC test (RF scratch): {roc_auc_score(y_te, rf_scratch.predict_proba(X_te_s)[:,1]):.4f}')

---
## 11. Pipeline de Comparación de Modelos con sklearn

In [ ]:
from itertools import product as iproduct

def pipeline_ensambles(X_tr, y_tr, X_te, y_te,
                        n_estimators_list=(50, 100, 200),
                        max_features_list=('sqrt', 'log2', 0.5),
                        max_depth_list=(None, 10, 20),
                        balance_list=(None, 'smote'),
                        cv=5, scoring='roc_auc',
                        models=('bagging', 'rf')):
    """
    Itera configuraciones de Bagging/RF con CV estratificado.
    Devuelve DataFrame ordenado por AUC_CV descendente.
    """
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    resultados = []
    for balance in balance_list:
        prep = PrepGener(scaler='standard', balance=balance)
        Xb, yb = prep.fit_resample(X_tr, y_tr)
        Xt = prep.transform(X_te)
        for n_est, mf, md, mdl in iproduct(
                n_estimators_list, max_features_list, max_depth_list, models):
            if mdl == 'bagging':
                base = DecisionTreeClassifier(max_depth=md)
                clf = BaggingClassifier(estimator=base, n_estimators=n_est,
                                        bootstrap=True, oob_score=True,
                                        n_jobs=-1, random_state=42)
            else:
                clf = RandomForestClassifier(n_estimators=n_est, max_features=mf,
                                             max_depth=md, oob_score=True,
                                             n_jobs=-1, random_state=42)
            cv_scores = cross_val_score(clf, Xb, yb, cv=skf,
                                        scoring=scoring, n_jobs=-1)
            clf.fit(Xb, yb)
            y_prob = clf.predict_proba(Xt)[:, 1]
            y_pred = clf.predict(Xt)
            resultados.append({
                'modelo': mdl, 'n_est': n_est, 'max_feat': mf,
                'max_depth': md, 'balance': balance,
                'AUC_CV': cv_scores.mean().round(4),
                'AUC_CV_std': cv_scores.std().round(4),
                'AUC_test': roc_auc_score(y_te, y_prob).round(4),
                'Acc_test': accuracy_score(y_te, y_pred).round(4),
                'OOB_err': round(1 - clf.oob_score_, 4)
            })
            del clf; gc.collect()
        del Xb, yb, Xt; gc.collect()

    return pd.DataFrame(resultados).sort_values('AUC_CV', ascending=False).reset_index(drop=True)


# Ejecución con grid reducido para demo
df_res = pipeline_ensambles(
    X_tr, y_tr, X_te, y_te,
    n_estimators_list=(50, 100),
    max_features_list=('sqrt', 'log2'),
    max_depth_list=(None, 10),
    balance_list=(None,),
    models=('bagging', 'rf')
)
print(df_res.head(10).to_string(index=False))

---
## 12. Diagnóstico del Mejor Modelo

In [ ]:
def entrenar_mejor(row, X_tr, y_tr, X_te, y_te):
    """Re-entrena el modelo top y genera todos los diagnósticos."""
    prep = PrepGener(scaler='standard', balance=row.balance)
    Xb, yb = prep.fit_resample(X_tr, y_tr)
    Xt = prep.transform(X_te)
    if row.modelo == 'bagging':
        base = DecisionTreeClassifier(max_depth=row.max_depth)
        clf = BaggingClassifier(estimator=base, n_estimators=row.n_est,
                                bootstrap=True, oob_score=True,
                                n_jobs=-1, random_state=42)
    else:
        clf = RandomForestClassifier(n_estimators=row.n_est,
                                     max_features=row.max_feat,
                                     max_depth=row.max_depth,
                                     oob_score=True, n_jobs=-1, random_state=42)
    clf.fit(Xb, yb)
    y_pred = clf.predict(Xt)
    y_prob = clf.predict_proba(Xt)[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    # Matriz de confusión
    ConfusionMatrixDisplay.from_predictions(y_te, y_pred, ax=axes[0],
                                             colorbar=False)
    axes[0].set_title('Matriz de Confusión (test)')
    # Importancia de variables (solo RF tiene feature_importances_)
    if row.modelo == 'rf':
        imp = pd.Series(clf.feature_importances_,
                        index=credit.drop('target',axis=1).columns)
        imp.nlargest(10).sort_values().plot.barh(ax=axes[1])
        axes[1].set_title('Top-10 Importancia (Gini)')
    else:
        axes[1].axis('off')
        axes[1].text(0.5, 0.5, 'Bagging: sin importancia directa',
                     ha='center', va='center', fontsize=12)
    plt.tight_layout(); plt.show()
    print(classification_report(y_te, y_pred))
    print(f'AUC: {roc_auc_score(y_te, y_prob):.4f}  |  OOB error: {1-clf.oob_score_:.4f}')
    del clf, Xb, yb, Xt; gc.collect()


mejor = df_res.iloc[0]
print(f'Mejor config: {mejor[["modelo","n_est","max_feat","max_depth","AUC_CV"]].to_dict()}')
entrenar_mejor(mejor, X_tr, y_tr, X_te, y_te)

---
## 13. Curva de Error OOB vs. Número de Árboles

In [ ]:
def curva_oob(X, y, n_max=300, step=10, max_features='sqrt'):
    """Error OOB en función del número de árboles — RF y Bagging."""
    ns = np.arange(step, n_max + 1, step)
    oob_rf, oob_bag = [], []
    for n in ns:
        rf = RandomForestClassifier(n_estimators=n, max_features=max_features,
                                     oob_score=True, n_jobs=-1, random_state=42)
        rf.fit(X, y)
        oob_rf.append(1 - rf.oob_score_)
        bg = BaggingClassifier(DecisionTreeClassifier(), n_estimators=n,
                               oob_score=True, n_jobs=-1, random_state=42)
        bg.fit(X, y)
        oob_bag.append(1 - bg.oob_score_)
        del rf, bg; gc.collect()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(ns, oob_rf, label='Random Forest', lw=2)
    ax.plot(ns, oob_bag, label='Bagging', lw=2, linestyle='--')
    ax.set_xlabel('Número de árboles'); ax.set_ylabel('OOB Error')
    ax.set_title('Convergencia del Error OOB')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


prep2 = PrepGener(scaler='standard')
X_full, y_full = prep2.fit_resample(X_tr, y_tr)
curva_oob(X_full, y_full, n_max=200)
del X_full, y_full, prep2; gc.collect()

---
## 14. Importancia por Permutación (con RF from scratch)

In [ ]:
prep3 = PrepGener(scaler='standard')
Xb3, yb3 = prep3.fit_resample(X_tr, y_tr)
Xt3 = prep3.transform(X_te)

rf_imp = RandomForestScratch(n_estimators=100, max_features='sqrt', random_state=42)
rf_imp.fit(Xb3, yb3)
imp_perm = rf_imp.feature_importance_permutation(Xt3, y_te)

feat_names = credit.drop('target', axis=1).columns
imp_df = pd.Series(imp_perm, index=feat_names).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
imp_df.tail(15).plot.barh(ax=ax, color='steelblue')
ax.axvline(0, color='k', lw=0.8, linestyle='--')
ax.set_title('Importancia por Permutación (RF scratch) — Top 15')
ax.set_xlabel('Δ Accuracy al permutar'); plt.tight_layout(); plt.show()

del Xb3, yb3, Xt3, rf_imp, prep3; gc.collect()

---
## 15. Búsqueda de Hiperparámetros — RF

Grid fino sobre `mtry` (≡ `max_features`), `max_depth` y `min_samples_leaf`.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

prep4 = PrepGener(scaler='standard')
Xb4, yb4 = prep4.fit_resample(X_tr, y_tr)
Xt4 = prep4.transform(X_te)

param_dist = {
    'n_estimators':    randint(50, 300),
    'max_features':    ['sqrt', 'log2', 0.3, 0.5],
    'max_depth':       [None, 5, 10, 20, 30],
    'min_samples_leaf': randint(1, 20),
    'min_samples_split': randint(2, 20)
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42),
    param_dist, n_iter=40, scoring='roc_auc', cv=skf,
    refit=True, n_jobs=-1, random_state=42, verbose=0
)
search.fit(Xb4, yb4)
print('Mejores hiperparámetros:')
print(search.best_params_)
print(f'AUC CV: {search.best_score_:.4f}')
print(f'AUC test: {roc_auc_score(y_te, search.best_estimator_.predict_proba(Xt4)[:,1]):.4f}')

del Xb4, yb4, Xt4, prep4; gc.collect()

---
## 16. Validación Cruzada Estratificada — Comparativa Final

In [ ]:
from sklearn.tree import DecisionTreeClassifier as DTC

def cv_comparativa(X, y, cv=5, random_state=42):
    """Compara árbol simple vs Bagging vs RF mediante CV estratificado."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    best_p = search.best_params_
    clfs = {
        'Árbol (sin ensamblar)': DTC(max_depth=5, random_state=42),
        'Bagging (n=100)': BaggingClassifier(DTC(), n_estimators=100,
                            oob_score=True, n_jobs=-1, random_state=42),
        'RF (n=100, sqrt)': RandomForestClassifier(n_estimators=100,
                            max_features='sqrt', oob_score=True,
                            n_jobs=-1, random_state=42),
        'RF (tuneado)': RandomForestClassifier(**best_p,
                            oob_score=True, n_jobs=-1, random_state=42)
    }
    rows = []
    for nombre, clf in clfs.items():
        auc = cross_val_score(clf, X, y, cv=skf,
                              scoring='roc_auc', n_jobs=-1)
        acc = cross_val_score(clf, X, y, cv=skf,
                              scoring='accuracy', n_jobs=-1)
        rows.append({'Modelo': nombre,
                     'AUC_mean': auc.mean().round(4),
                     'AUC_std': auc.std().round(4),
                     'Acc_mean': acc.mean().round(4),
                     'Acc_std': acc.std().round(4)})
        del clf; gc.collect()
    return pd.DataFrame(rows).sort_values('AUC_mean', ascending=False)


prep5 = PrepGener(scaler='standard')
Xall, yall = prep5.fit_resample(X, y)
df_cv = cv_comparativa(Xall, yall)
print(df_cv.to_string(index=False))

# Gráfica
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(df_cv.Modelo, df_cv.AUC_mean, xerr=df_cv.AUC_std,
        capsize=4, color='steelblue', alpha=0.8)
ax.set_xlabel('AUC (5-fold CV)'); ax.set_title('Comparativa de Ensambles')
ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()
del Xall, yall, prep5; gc.collect()

---
## 17. Ejemplo con Iris (clasificación multiclase)

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
Xi, yi = iris.data.values, iris.target.values
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.2,
                                               stratify=yi, random_state=42)

rf_iris = RandomForestClassifier(n_estimators=100, oob_score=True,
                                  n_jobs=-1, random_state=42)
rf_iris.fit(Xi_tr, yi_tr)
y_hat_iris = rf_iris.predict(Xi_te)
print(classification_report(yi_te, y_hat_iris,
      target_names=iris.target_names))
print(f'OOB error: {1-rf_iris.oob_score_:.4f}')

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(yi_te, y_hat_iris,
    display_labels=iris.target_names, ax=ax, colorbar=False)
ax.set_title('Iris — RF Confusion Matrix')
plt.tight_layout(); plt.show()
del rf_iris; gc.collect()

---
---
# Sección R — Implementación Equivalente

> Las celdas siguientes replican todos los análisis anteriores en R.
> Requiere `rpy2`. En Colab: `!pip install rpy2==3.5.1 && %reload_ext rpy2.ipython`

In [ ]:
!pip install rpy2==3.5.1 -q
%reload_ext rpy2.ipython
%config IPCompleter.greedy=True

In [ ]:
%%R
pkgs <- c('dplyr','rpart','ipred','randomForest','caret',
          'ModelMetrics','ggplot2','ISLR')
invisible(lapply(pkgs, function(p)
  if(!requireNamespace(p, quietly=TRUE)) install.packages(p)))
invisible(lapply(pkgs, library, character.only=TRUE))
cat('Paquetes R listos\n')

In [ ]:
%%R
credit <- read.table('CursoDF/M3/Datos/credit.txt')
credit <- mutate(credit, V21 = as.factor(V21))
cred_nam <- c('checking_status','duration','credit_history','purpose',
              'credit_amount','savings','employment','installment_rate',
              'personal_status','debtors','residence_since','property',
              'age','other_plans','housing','existing_credits','job',
              'dependents','telephone','foreign_worker','target')
names(credit) <- make.names(cred_nam)
set.seed(42)
idx <- sample(nrow(credit), floor(0.7 * nrow(credit)))
cr_train <- credit[idx,]; cr_test  <- credit[-idx,]
cat(sprintf('Train: %d  |  Test: %d\n', nrow(cr_train), nrow(cr_test)))

In [ ]:
%%R
# --- Bagging (ipred) ---
set.seed(123)
bag_model <- bagging(formula = target ~ ., data = cr_train,
                     nbagg = 100, coob = TRUE)
cat(sprintf('OOB error (Bagging): %.4f\n', bag_model$err))
pred_bag <- predict(bag_model, newdata = cr_test, type = 'class')
cm_bag <- confusionMatrix(pred_bag, cr_test$target)
cat(sprintf('Accuracy Bagging: %.4f\n', cm_bag$overall['Accuracy']))
pred_prob_bag <- predict(bag_model, newdata = cr_test, type = 'prob')
cat(sprintf('AUC Bagging: %.4f\n',
    auc(actual = ifelse(cr_test$target=='1',1,0),
        predicted = pred_prob_bag[,'1'])))

In [ ]:
%%R
# --- Random Forest ---
set.seed(256)
rf_model <- randomForest(formula = target ~ ., data = cr_train,
                          ntree = 200, mtry = floor(sqrt(ncol(cr_train)-1)),
                          importance = TRUE, keep.forest = TRUE)
print(rf_model)
plot(rf_model, main = 'Error OOB vs Número de árboles')
legend('topright', legend = colnames(rf_model$err.rate),
       col = 1:ncol(rf_model$err.rate), lty = 1, bty = 'n')

In [ ]:
%%R
# Importancia de variables
varImpPlot(rf_model, main = 'Importancia de Variables (RF)')

# Métricas test
pred_rf  <- predict(rf_model, newdata = cr_test, type = 'class')
prob_rf  <- predict(rf_model, newdata = cr_test, type = 'prob')
cm_rf    <- confusionMatrix(pred_rf, cr_test$target)
print(cm_rf)
cat(sprintf('AUC RF: %.4f\n',
    auc(actual = ifelse(cr_test$target=='1',1,0),
        predicted = prob_rf[,'1'])))

In [ ]:
%%R
# --- Búsqueda de hiperparámetros: grid sobre mtry, nodesize, sampsize ---
mtry_vals    <- seq(3, floor(ncol(cr_train) * 0.7), by = 2)
nodesize_vals <- c(3, 5, 8)
sampsize_vals <- floor(nrow(cr_train) * c(0.6, 0.7, 0.8))
hyper_grid <- expand.grid(mtry=mtry_vals, nodesize=nodesize_vals,
                           sampsize=sampsize_vals)
oob_err <- numeric(nrow(hyper_grid))
set.seed(42)
for (i in seq_len(nrow(hyper_grid))) {
  m <- randomForest(formula = target ~ ., data = cr_train, ntree = 100,
                    mtry       = hyper_grid$mtry[i],
                    nodesize   = hyper_grid$nodesize[i],
                    sampsize   = hyper_grid$sampsize[i])
  oob_err[i] <- m$err.rate[nrow(m$err.rate), 'OOB']
}
hyper_grid$oob_err <- oob_err
opt <- hyper_grid[which.min(oob_err), ]
cat('Mejor configuración:\n'); print(opt)

In [ ]:
%%R
# --- Modelo RF óptimo + CV con caret ---
ctrl <- trainControl(method='cv', number=5, classProbs=TRUE,
                     summaryFunction=twoClassSummary, savePredictions='final')
levels(cr_train$target) <- c('malo','bueno')
levels(cr_test$target)  <- c('malo','bueno')
set.seed(42)
rf_cv <- train(target ~ ., data=cr_train, method='rf', metric='ROC',
               trControl=ctrl,
               tuneGrid=data.frame(mtry=opt$mtry),
               ntree=200, nodesize=opt$nodesize)
print(rf_cv)
pred_cv  <- predict(rf_cv, newdata=cr_test)
prob_cv  <- predict(rf_cv, newdata=cr_test, type='prob')
cm_cv    <- confusionMatrix(pred_cv, cr_test$target)
print(cm_cv)
cat(sprintf('AUC RF (caret CV): %.4f\n',
    auc(actual=ifelse(cr_test$target=='bueno',1,0),
        predicted=prob_cv[,'bueno'])))

In [ ]:
%%R
# --- Ejemplo IRIS multiclase ---
set.seed(42)
idx_i <- sample(nrow(iris), floor(0.8*nrow(iris)))
iris_tr <- iris[idx_i,]; iris_te <- iris[-idx_i,]
rf_iris <- randomForest(Species ~., data=iris_tr,
                        xtest=iris_te[,-5], ytest=iris_te$Species,
                        ntree=100, importance=TRUE)
print(rf_iris$test$confusion)
varImpPlot(rf_iris, main='Importancia — IRIS')
plot(rf_iris, main='OOB Error — IRIS')